In [ ]:
class Document:
    def __init__(self, titre, auteur, date, url, texte):
        self.titre = titre
        self.auteur = auteur
        self.date = date
        self.url = url
        self.texte = texte

    def infos(self):
        print("Titre : ", self.titre)
        print("Auteur : ", self.auteur)
        print("date : ", self.date)
        print("url : ", self.url)
        print("texte : ", self.texte)

    def __str__(self):
        return self.titre

class HackerNewsDocument(Document):
    def __init__(self, titre, auteur, date, url, texte, nb_comments):
        super.__init__(titre, auteur, date, url, texte)
        self.nb_comments = nb_comments

In [ ]:
# Objectif : construire un base de données documentaire sur un sujet choisi.
# Deux options : requêter l'API sur le sujet choisi
# Possibilité de choisir une date pour construire la bdd

import urllib.parse
import urllib.request
import json
import math
from datetime import datetime, timezone
import html
import pandas as pd

def hits_to_list(hits):
    # Le champ contenant le contenu textuel est story_text. Il est absent des
    # documents qui ne pointent que vers un lien extérieur : dans ce cas on se
    # rabat sur le titre (champ title), toujours présent.
    for hit in hits: # Dans un hit se trouve toutes les données qui nous intéresse
        texte = hit.get("story_text")
        if not texte:
            texte = hit.get("title", "")
        # le texte contient du HTML : <p> pour les paragraphes et des caractères
        # échappés comme &#x27; pour l'apostrophe
        texte = html.unescape(texte).replace("<p>", " ").strip()
        
        auteur = hit.get("author")
        titre = hit.get("title")
        date = hit.get("created_at")
        url = hit.get("url")

        doc = Document(titre, auteur, datetime.fromisoformat(date.replace("Z", "+00:00")), url, texte)
        
        docs.append(doc)
        origines.append("hackernews")


    return


def get_json(url):
    try:
        with urllib.request.urlopen(url) as reponse:
            d_hn = json.load(reponse)

    except Exception as e:
        print("Erreur récupération json : ", e)
        raise
    return d_hn

# Version sans date
def requete_hacker_news(nb_hits_page, theme):

    url = "https://hn.algolia.com/api/v1/search_by_date?" + urllib.parse.urlencode(
        {"query": theme, "tags": "story", "hitsPerPage": nb_hits_page})

    print("requete: ", url)

    d_hn = get_json(url)
    print(d_hn)

    nb_hits = d_hn["nbHits"]
    print("Nombre de hits trouvés : ", nb_hits)

    hits = d_hn["hits"]
    print("enregistrement des 1000 premiers hits")
    hits_to_list(hits)

    if nb_hits > 1000:
        nb_iterations = math.ceil(nb_hits / nb_hits_page) #Nombre d'itérations à effectuer pour requêter la totalité des données issues de la requête (max 1000 par requête)
        print("Nombre de requetes pour tout stocker : ", nb_iterations)
        for i in range(1,nb_iterations):

            nv_debut = hits[-1]["created_at_i"]
            nv_date = "created_at_i<" + str(nv_debut)
            nv_url = url + "&numericFilters=" + urllib.parse.quote(nv_date, safe=",<>=")

            d_hn = get_json(nv_url)
            
            hits = d_hn["hits"]
            hits_to_list(hits)
            print("enregistrement jusqu'aux  ", i+1, "000ème hits")
            print("total documents : ", len(docs))

    return

theme = "surveillance"
docs = []
origines = []

# Limite de l'API hackerNews : 10 000 requêtes par heure
requete_hacker_news(1000, "surveillance")


df = pd.DataFrame({"id": range(len(docs)),
                   "texte": docs,
                   "origine": origines})

df.to_csv("corpus.csv", sep="\t", index=False)

In [3]:
print(docs[1].infos())

Titre :  US university students tell surveillance cameras to Flock off
Auteur :  beardyw
date :  2026-11-13 14:11:56+00:00
url :  https://www.theguardian.com/technology/ng-interactive/2026/nov/13/universities-drop-flock-camera-contracts
texte :  US university students tell surveillance cameras to Flock off
None


In [4]:
id2doc = dict()
print(len(id2doc))

def docs_to_dict(docs, dictionnaire):
    for doc in docs:
        cle = str(doc.date) + str(doc.titre)
        if cle not in dictionnaire.keys():
            dictionnaire[cle]= doc
    return dictionnaire

id2doc = docs_to_dict(docs, id2doc)
print(len(id2doc))

0
8820


In [ ]:
class Author:
    def __init__(self, name, ndoc, production):
        self.name = name
        self.ndoc = ndoc if ndoc is not None else 0
        self.production = production if production is not None else []

    def add(self, dict_docs):
        for cle, valeur in dict_docs.items():
            if valeur.auteur == self.name and cle not in self.production:
                self.production.append(cle)
                self.ndoc = len(self.production)

    def __str__(self):
        return self.name

    def nb_docs(self):
        return self.ndoc

    #def mean_len_docs(self):
    

In [ ]:
def auteur_to_dict(obj_auteur, dict_auteurs):
    if obj_auteur.name not in dict_auteurs.keys():
        dict_auteurs[obj_auteur.name] = obj_auteur
    return(dict_auteurs)

In [7]:
a = Author("paulpauper", ndoc=None, production=None)

a.add(id2doc)

In [8]:
id2aut = dict()

def creer_dict_auteurs(dict_docs, auteurs):
    for cle, valeur in dict_docs.items():
        a = valeur.auteur
        if a not in auteurs.keys():
            auteurs[a] = Author(a, ndoc=None, production=None)
    return auteurs

id2aut = creer_dict_auteurs(id2doc, id2aut)

In [ ]:
class Corpus:
    def __init__(self, nom, dict_auteurs, dict_docs):
        self.nom = nom
        self.authors = dict_auteurs
        self.id2doc = dict_docs
        self.ndoc = int(len(id2doc))
        self.naut = int(len(dict_auteurs))
    